# S&P 500 European Options Pricing Model

## Importing Libraries
YFinance is a library used to get market data from Yahoo Finance. The data downloaded is 2 years' worth of historical data. Pandas has been used to handle and analyse data. Numpy has been used to carry out mathematical calculations. Scipy provided the CDF of the normal distribution. Datetime has been used to obtain the current date so that the US treasury yield data updates automatically to the most recent values available whenever the program is run.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import scipy as sp
import fractions as fr
import datetime as dt

## S&P 500 Historical Data
2 years of historical S&P 500 data have been downloaded; a longer period means that there is a larger sample of data to estimate volatility from. The most recent closing index level whenever the program is run is selected and used as S. The continuous dividend yield, q, is assumed to be 0.011, which at the time of writing is the most recent value for the S&P 500.

In [ ]:
df_index = yf.download('^GSPC', period='2y')
df_close = df_index['Close']
S = df_close.iloc[-1,0]
q = 0.011 #https://www.dividenddata.co.uk/sp500-dividend-yields.py
print(f'The most recent closing price from this data is {S}')
print(f'{q} is the continuous dividend yield, which has been estimated from the most recently published S&P 500 dividend yield. For the model, assume this stays constant.')

## Estimating Volatility
The annualised historical volatility calculated from the 2 years of S&P 500 data is used as an estimate for future volatility. It is only an estimate because volatility can change over time and can depend on a lot of factors that can be hard to predict or unexpected; this program assumes that historical volatility gives a good indication of future volatility. The annualised historical volatility is calculated by multiplying the standard deviation of the logarithmic returns from the downloaded data by $\sqrt{252}$, representing the approximately 252 trading days per year. Log returns are used in order to stay consistent with the assumptions made for the Black-Scholes model.

In [ ]:
df_returns = np.log(df_close/df_close.shift(1)).dropna()
sigma = (df_returns.std()).iloc[0] * np.sqrt(252)
print(f'{sigma} is our estimate of the annualised volatility.')


## User Inputs
The user inputs the option type (call or put) and an error message is displayed if the user does not enter either call or put, and they are prompted to enter another input. Then the user enters the maturity time in years, and an error message is displayed if the number is not between 1/12 and 30 or if the input is not a number and the user can try another input. After this has been satisfied, the user inputs the strike price in index points, and an error message is displayed if it is not positive or if it is not a number, and they can try another input.

In [ ]:
allowed_type = ['call', 'Call', 'Put', 'put']
while True: #user enters call or put
    option_type = input('If the option is a call option, enter call. If the option is a put option, enter put.')
    if option_type in allowed_type:
        break
    else:
        print('Please enter either call or put.')

while True: #user enters maturity
    try:
        T = float(fr.Fraction(input('Enter the maturity time of the option in years.')))
    except ValueError:
        print('Please enter a number')
        continue
    if T>=1/12 and T<=30:
        break
    else:
        print('Please enter a number of years between 1/12 and 30.')

while True: #user enters strike price
    try:
        k = float(input('Enter the strike price'))
    except ValueError:
        print('Please enter a number')
        continue
    if k>0:
        break
    else:
        print('Please enter a positive number.')

## Approximating r
To approximate r, the risk-free interest rate, the most recent data from the US daily treasury rates is downloaded every time the program runs, ensuring that the approximation is accurate to when the user runs the program. The datetime library was used in the url so that the data updates automatically whenever the program runs. US treasury securities are not risk free as required by Black-Scholes, so this is only an approximation. However, it has been chosen since it is considered to be very low risk, so it makes for a good approximation. The data has the interest rates for maturities between 1 month and 30 years, and since the data is discrete, the program uses linear interpolation to approximate the interest rate if the input for T is not one of the maturities listed in the dataset.

In [ ]:
datenow = dt.date.today().year
treasury = f"https://home.treasury.gov/resource-center/data-chart-center/interest-rates/daily-treasury-rates.csv/{datenow}/all?type=daily_treasury_yield_curve&field_tdr_date_value={datenow}&page&_format=csv" #US daily treasury rates are low risk so base estimation off this
df_treasury = pd.read_csv(treasury).head(1)
print(df_treasury)
maturities = np.array([1/12, 1/8, 1/6, 1/4, 1/3, 1/2, 1, 2, 3, 5, 7, 10, 20, 30]) #in years
rates = (df_treasury.iloc[0,1:]).to_numpy(dtype=np.float64) / 100
r = np.interp(T, maturities, rates)
print(f'{r:.4f} is the approximated annualised risk free interest rate')

## Calculating Theoretical Option Price
To calculate the theoretical option price in index points, the Black-Scholes model with a continuous dividend yield has been implemented. Both the theoretical call price and put price are calculated and stored under c_price and p_price respectively so that the Put-Call Parity Equation can be tested in the next section. The program outputs the price in S&P 500 index points corresponding to the selected option type.

In [ ]:
d_1 = (np.log(S/k) + T*(r - q + ((sigma)**2)/2))/(sigma*np.sqrt(T))
d_2 = d_1 - sigma*np.sqrt(T)
c_price = S*np.exp(-q*T)*sp.stats.norm.cdf(d_1) - k*np.exp(-r*T)*sp.stats.norm.cdf(d_2)
p_price = k*np.exp(-r*T)*sp.stats.norm.cdf(-d_2) - S*np.exp(-q*T)*sp.stats.norm.cdf(-d_1)

if option_type == 'call' or option_type == 'Call':
    print(f'{c_price:.2f} index points is the call option price.')
else:
    print(f'{p_price:.2f} index points is the put option price.')

print(f"S = {S}")
print(f"K = {k}")
print(f"T = {T}")
print(f"r = {r}")
print(f"q = {q}")
print(f"sigma = {sigma}")

## Testing Put-Call Parity
The Put-Call Parity Equation checks for consistency by using a model-free relationship between European call and put prices that have the same maturity, underlying asset and strike price. np.isclose() has a small margin of error, so this has been used to account for any rounding errors. For this model, the parity equation has been satisfied.

In [ ]:
lhs = c_price - p_price
rhs = S*np.exp(-q*T)-k*np.exp(-r*T)
if np.isclose(lhs, rhs) == True:
    print('The parity equation has been satisfied')
else:
    print('The parity equation has not been satisfied')